## Load the Restaurant Dataset

### Create the Unity Catalog Structure

In [ ]:
%sql

CREATE SCHEMA IF NOT EXISTS food_delivery_lab.bronze
COMMENT 'Raw ingested delivery data';

CREATE VOLUME IF NOT EXISTS food_delivery_lab.bronze.raw_files
COMMENT 'Landing zone for incoming CSV files';

### Write CSV Files to UC Volume

In [ ]:
import requests

# Define the current catalog
catalog_name = "food_delivery_lab"

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/bronze/raw_files"

# List of files to download
files = ["orders.csv", "restaurants.csv", "payments.csv", "executives.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/Databricks-Job-Ready-Projects/refs/heads/main/Databricks-Fundamentals/Restaurant_Data/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

print("CSV Files written successfully")

### Load into Bronze Delta Tables

In [ ]:
def load_csv(path, table):
    (spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(path)
        .write.mode("overwrite")
        .saveAsTable(table))

base = "/Volumes/food_delivery_lab/bronze/raw_files"

load_csv(f"{base}/orders.csv",      "food_delivery_lab.bronze.orders")
load_csv(f"{base}/restaurants.csv", "food_delivery_lab.bronze.restaurants")
load_csv(f"{base}/executives.csv",     "food_delivery_lab.bronze.executives")
load_csv(f"{base}/payments.csv",    "food_delivery_lab.bronze.payments")

print("All tables loaded")